# Ship Reduction vs Sentiment Correlation Analysis
## Independent of Stock Performance

This notebook analyzes the correlation between reductions in ship arrivals and sentiment surrounding geopolitical crises, without considering stock market performance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

plt.rcParams['figure.figsize'] = (16, 8)
plt.rcParams['font.size'] = 11

## 1. Load and Prepare Data

In [ ]:
# Load shipping data
arrivals_df = pd.read_csv('Data/Portwatch_Shipment_Data/arrivals-of-ships.csv')
arrivals_df['DateTime'] = pd.to_datetime(arrivals_df['DateTime'])
arrivals_df.set_index('DateTime', inplace=True)
arrivals_df['Total'] = arrivals_df[['Container', 'Dry Bulk', 'General Cargo', 'Roll-on/roll-off', 'Tanker']].sum(axis=1)

# Load sentiment data
sentiment_daily = pd.read_csv('Data/crisis_sentiment_daily.csv')
sentiment_daily['date'] = pd.to_datetime(sentiment_daily['date'])
sentiment_daily.set_index('date', inplace=True)

print(f"Shipping data: {arrivals_df.index.min()} to {arrivals_df.index.max()}")
print(f"Sentiment data: {sentiment_daily.index.min()} to {sentiment_daily.index.max()}")
print(f"\nCrisis events: {sentiment_daily['crisis_event'].nunique()}")

## 2. Calculate Ship Reduction Metrics

We'll calculate multiple metrics to capture ship reductions:
- Percentage change from baseline (prior year average)
- Absolute reduction in ship counts
- Rolling average changes

In [ ]:
# Calculate ship reduction metrics
arrivals_df['pct_change_from_prior_year'] = (
    (arrivals_df['7-day Moving Average'] - arrivals_df['Prior Year: 7-day Moving Average']) / 
    arrivals_df['Prior Year: 7-day Moving Average'] * 100
)

arrivals_df['absolute_reduction'] = (
    arrivals_df['Prior Year: 7-day Moving Average'] - arrivals_df['7-day Moving Average']
)

# Calculate rolling statistics
arrivals_df['rolling_30d_mean'] = arrivals_df['Total'].rolling(window=30, min_periods=1).mean()
arrivals_df['rolling_30d_std'] = arrivals_df['Total'].rolling(window=30, min_periods=1).std()

# Z-score to identify significant deviations
arrivals_df['z_score'] = (
    (arrivals_df['Total'] - arrivals_df['rolling_30d_mean']) / arrivals_df['rolling_30d_std']
)

print("Ship reduction metrics calculated:")
print(arrivals_df[['Total', 'pct_change_from_prior_year', 'absolute_reduction', 'z_score']].describe())

## 3. Merge Shipping and Sentiment Data

In [ ]:
# Merge datasets on date
merged_df = arrivals_df.join(sentiment_daily, how='inner')

print(f"Merged dataset: {len(merged_df)} observations")
print(f"Date range: {merged_df.index.min()} to {merged_df.index.max()}")
print(f"\nColumns: {merged_df.columns.tolist()}")

# Display sample
merged_df[['Total', 'pct_change_from_prior_year', 'sentiment_mean', 'risk_level', 'crisis_event']].head(10)

## 4. Correlation Analysis: Ship Reduction vs Sentiment

In [ ]:
# Calculate correlations
correlations = {}

# Remove NaN values for correlation
clean_df = merged_df[['pct_change_from_prior_year', 'absolute_reduction', 'z_score', 
                       'sentiment_mean', 'confidence_mean']].dropna()

# Pearson correlation
correlations['pct_change_vs_sentiment'] = stats.pearsonr(
    clean_df['pct_change_from_prior_year'], 
    clean_df['sentiment_mean']
)

correlations['absolute_reduction_vs_sentiment'] = stats.pearsonr(
    clean_df['absolute_reduction'], 
    clean_df['sentiment_mean']
)

correlations['z_score_vs_sentiment'] = stats.pearsonr(
    clean_df['z_score'], 
    clean_df['sentiment_mean']
)

# Spearman correlation (non-parametric)
correlations['spearman_pct_vs_sentiment'] = stats.spearmanr(
    clean_df['pct_change_from_prior_year'], 
    clean_df['sentiment_mean']
)

print("=" * 70)
print("CORRELATION ANALYSIS: Ship Reduction vs Sentiment")
print("=" * 70)
print(f"\nSample size: {len(clean_df)} observations\n")

for key, (corr, pval) in correlations.items():
    sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else "ns"
    print(f"{key}:")
    print(f"  Correlation: {corr:.4f}")
    print(f"  P-value: {pval:.6f} {sig}")
    print()

## 5. Visualization: Ship Reduction vs Sentiment

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# 1. Scatter: Percentage change vs Sentiment
ax1 = axes[0, 0]
scatter1 = ax1.scatter(clean_df['pct_change_from_prior_year'], clean_df['sentiment_mean'], 
                       c=clean_df['confidence_mean'], cmap='viridis', alpha=0.6, s=100)
ax1.axhline(y=0, color='red', linestyle='--', alpha=0.3, label='Neutral Sentiment')
ax1.axvline(x=0, color='blue', linestyle='--', alpha=0.3, label='No Change from Prior Year')
z = np.polyfit(clean_df['pct_change_from_prior_year'], clean_df['sentiment_mean'], 1)
p = np.poly1d(z)
ax1.plot(clean_df['pct_change_from_prior_year'], p(clean_df['pct_change_from_prior_year']), 
         "r--", alpha=0.8, linewidth=2, label=f'Trend (r={correlations["pct_change_vs_sentiment"][0]:.3f})')
ax1.set_xlabel('% Change in Ships from Prior Year', fontsize=12, fontweight='bold')
ax1.set_ylabel('Sentiment Score', fontsize=12, fontweight='bold')
ax1.set_title('Ship Reduction vs Sentiment\n(% Change from Prior Year)', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)
plt.colorbar(scatter1, ax=ax1, label='Confidence')

# 2. Scatter: Absolute reduction vs Sentiment
ax2 = axes[0, 1]
scatter2 = ax2.scatter(clean_df['absolute_reduction'], clean_df['sentiment_mean'], 
                       c=clean_df['confidence_mean'], cmap='plasma', alpha=0.6, s=100)
ax2.axhline(y=0, color='red', linestyle='--', alpha=0.3)
ax2.axvline(x=0, color='blue', linestyle='--', alpha=0.3)
z2 = np.polyfit(clean_df['absolute_reduction'], clean_df['sentiment_mean'], 1)
p2 = np.poly1d(z2)
ax2.plot(clean_df['absolute_reduction'], p2(clean_df['absolute_reduction']), 
         "r--", alpha=0.8, linewidth=2, label=f'Trend (r={correlations["absolute_reduction_vs_sentiment"][0]:.3f})')
ax2.set_xlabel('Absolute Reduction in Ships', fontsize=12, fontweight='bold')
ax2.set_ylabel('Sentiment Score', fontsize=12, fontweight='bold')
ax2.set_title('Ship Reduction vs Sentiment\n(Absolute Numbers)', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)
plt.colorbar(scatter2, ax=ax2, label='Confidence')

# 3. Box plot: Sentiment by risk level with ship reduction
ax3 = axes[1, 0]
risk_order = ['low', 'medium', 'high', 'critical']
risk_data = [clean_df[clean_df.index.isin(merged_df[merged_df['risk_level']==r].index)]['pct_change_from_prior_year'].values 
             for r in risk_order if r in merged_df['risk_level'].values]
risk_labels = [r for r in risk_order if r in merged_df['risk_level'].values]
bp = ax3.boxplot(risk_data, labels=risk_labels, patch_artist=True)
colors = ['lightgreen', 'yellow', 'orange', 'red']
for patch, color in zip(bp['boxes'], colors[:len(risk_labels)]):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
ax3.axhline(y=0, color='blue', linestyle='--', alpha=0.5, label='No Change')
ax3.set_xlabel('Risk Level', fontsize=12, fontweight='bold')
ax3.set_ylabel('% Change in Ships from Prior Year', fontsize=12, fontweight='bold')
ax3.set_title('Ship Reduction by Risk Level', fontsize=14, fontweight='bold')
ax3.legend()
ax3.grid(True, alpha=0.3, axis='y')

# 4. Heatmap: Correlation matrix
ax4 = axes[1, 1]
corr_matrix = clean_df[['pct_change_from_prior_year', 'absolute_reduction', 'z_score', 
                         'sentiment_mean', 'confidence_mean']].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8}, ax=ax4)
ax4.set_title('Correlation Matrix\n(Ship Metrics vs Sentiment)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('ship_reduction_sentiment_correlation.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nVisualization saved as 'ship_reduction_sentiment_correlation.png'")

## 6. Crisis-Specific Analysis

In [ ]:
# Analyze each crisis event separately
crisis_correlations = []

for crisis in merged_df['crisis_event'].unique():
    crisis_data = merged_df[merged_df['crisis_event'] == crisis][[
        'pct_change_from_prior_year', 'absolute_reduction', 'sentiment_mean'
    ]].dropna()
    
    if len(crisis_data) >= 3:  # Need at least 3 points for correlation
        corr_pct, pval_pct = stats.pearsonr(
            crisis_data['pct_change_from_prior_year'], 
            crisis_data['sentiment_mean']
        )
        corr_abs, pval_abs = stats.pearsonr(
            crisis_data['absolute_reduction'], 
            crisis_data['sentiment_mean']
        )
        
        crisis_correlations.append({
            'crisis': crisis,
            'n_observations': len(crisis_data),
            'corr_pct_change': corr_pct,
            'pval_pct_change': pval_pct,
            'corr_absolute': corr_abs,
            'pval_absolute': pval_abs,
            'avg_sentiment': crisis_data['sentiment_mean'].mean(),
            'avg_reduction_pct': crisis_data['pct_change_from_prior_year'].mean()
        })

crisis_corr_df = pd.DataFrame(crisis_correlations)
crisis_corr_df = crisis_corr_df.sort_values('corr_pct_change', ascending=False)

print("=" * 100)
print("CRISIS-SPECIFIC CORRELATIONS: Ship Reduction vs Sentiment")
print("=" * 100)
print(crisis_corr_df.to_string(index=False))
print("\n* Negative correlation means: more ship reduction → more negative sentiment")
print("* Positive correlation means: more ship reduction → less negative sentiment (unexpected)")

## 7. Time-Lagged Correlation Analysis

Check if sentiment leads or lags ship reductions

In [ ]:
# Calculate lagged correlations
lags = range(-14, 15)  # -14 to +14 days
lagged_correlations = []

for lag in lags:
    if lag < 0:
        # Sentiment leads (negative lag)
        temp_df = pd.DataFrame({
            'sentiment': merged_df['sentiment_mean'].shift(-lag),
            'ship_reduction': merged_df['pct_change_from_prior_year']
        }).dropna()
    else:
        # Ship reduction leads (positive lag)
        temp_df = pd.DataFrame({
            'sentiment': merged_df['sentiment_mean'],
            'ship_reduction': merged_df['pct_change_from_prior_year'].shift(lag)
        }).dropna()
    
    if len(temp_df) > 0:
        corr, pval = stats.pearsonr(temp_df['sentiment'], temp_df['ship_reduction'])
        lagged_correlations.append({'lag': lag, 'correlation': corr, 'pvalue': pval})

lag_df = pd.DataFrame(lagged_correlations)

# Plot lagged correlations
fig, ax = plt.subplots(figsize=(14, 6))
colors = ['red' if p < 0.05 else 'gray' for p in lag_df['pvalue']]
ax.bar(lag_df['lag'], lag_df['correlation'], color=colors, alpha=0.7)
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.8)
ax.axvline(x=0, color='blue', linestyle='--', linewidth=2, label='No Lag')
ax.set_xlabel('Lag (days)\n← Sentiment Leads | Ship Reduction Leads →', fontsize=12, fontweight='bold')
ax.set_ylabel('Correlation Coefficient', fontsize=12, fontweight='bold')
ax.set_title('Time-Lagged Correlation: Sentiment vs Ship Reduction\n(Red bars = significant at p<0.05)', 
             fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('lagged_correlation_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

# Find optimal lag
max_corr_idx = lag_df['correlation'].abs().idxmax()
optimal_lag = lag_df.loc[max_corr_idx]

print("\n" + "=" * 70)
print("TIME-LAGGED CORRELATION ANALYSIS")
print("=" * 70)
print(f"\nOptimal lag: {optimal_lag['lag']} days")
print(f"Correlation at optimal lag: {optimal_lag['correlation']:.4f}")
print(f"P-value: {optimal_lag['pvalue']:.6f}")

if optimal_lag['lag'] < 0:
    print(f"\nInterpretation: Sentiment LEADS ship reduction by {abs(optimal_lag['lag'])} days")
elif optimal_lag['lag'] > 0:
    print(f"\nInterpretation: Ship reduction LEADS sentiment by {optimal_lag['lag']} days")
else:
    print(f"\nInterpretation: Sentiment and ship reduction are CONTEMPORANEOUS")

## 8. Summary Statistics by Risk Level

In [ ]:
# Group by risk level
risk_summary = merged_df.groupby('risk_level').agg({
    'sentiment_mean': ['mean', 'std', 'count'],
    'pct_change_from_prior_year': ['mean', 'std'],
    'absolute_reduction': ['mean', 'std'],
    'Total': ['mean', 'std']
}).round(3)

print("\n" + "=" * 100)
print("SUMMARY STATISTICS BY RISK LEVEL")
print("=" * 100)
print(risk_summary)

# Statistical test: ANOVA for ship reduction across risk levels
risk_groups = [merged_df[merged_df['risk_level']==r]['pct_change_from_prior_year'].dropna().values 
               for r in merged_df['risk_level'].unique()]
f_stat, p_val = stats.f_oneway(*risk_groups)

print(f"\nANOVA Test: Ship reduction differs across risk levels?")
print(f"F-statistic: {f_stat:.4f}")
print(f"P-value: {p_val:.6f}")
if p_val < 0.05:
    print("Result: YES - Ship reduction significantly differs across risk levels")
else:
    print("Result: NO - No significant difference in ship reduction across risk levels")

## 9. Key Findings Summary

In [ ]:
print("\n" + "=" * 100)
print("KEY FINDINGS: SHIP REDUCTION vs SENTIMENT CORRELATION")
print("=" * 100)

print("\n1. OVERALL CORRELATION:")
print(f"   - Percentage change vs sentiment: r = {correlations['pct_change_vs_sentiment'][0]:.4f} (p = {correlations['pct_change_vs_sentiment'][1]:.6f})")
print(f"   - Absolute reduction vs sentiment: r = {correlations['absolute_reduction_vs_sentiment'][0]:.4f} (p = {correlations['absolute_reduction_vs_sentiment'][1]:.6f})")

print("\n2. INTERPRETATION:")
if correlations['pct_change_vs_sentiment'][0] < 0:
    print("   - NEGATIVE correlation: When ships decrease, sentiment becomes MORE negative")
    print("   - This suggests ship reductions coincide with worsening crisis sentiment")
else:
    print("   - POSITIVE correlation: When ships decrease, sentiment becomes LESS negative")
    print("   - This is unexpected and may indicate complex dynamics")

print("\n3. STATISTICAL SIGNIFICANCE:")
if correlations['pct_change_vs_sentiment'][1] < 0.05:
    print("   - The correlation IS statistically significant (p < 0.05)")
else:
    print("   - The correlation is NOT statistically significant (p >= 0.05)")

print("\n4. CRISIS-SPECIFIC PATTERNS:")
print(f"   - Analyzed {len(crisis_corr_df)} crisis events")
print(f"   - Strongest correlation: {crisis_corr_df.iloc[0]['crisis']} (r = {crisis_corr_df.iloc[0]['corr_pct_change']:.4f})")
print(f"   - Weakest correlation: {crisis_corr_df.iloc[-1]['crisis']} (r = {crisis_corr_df.iloc[-1]['corr_pct_change']:.4f})")

print("\n5. TEMPORAL DYNAMICS:")
if optimal_lag['lag'] < 0:
    print(f"   - Sentiment LEADS ship reduction by {abs(optimal_lag['lag'])} days")
    print("   - Negative news may cause shipping companies to reduce operations")
elif optimal_lag['lag'] > 0:
    print(f"   - Ship reduction LEADS sentiment by {optimal_lag['lag']} days")
    print("   - Shipping disruptions may drive negative news coverage")
else:
    print("   - Sentiment and ship reduction occur SIMULTANEOUSLY")
    print("   - Both respond to the same underlying crisis events")

print("\n" + "=" * 100)